In [1]:
import sys
print(sys.executable)

C:\Users\HP\GeoPulse--Hyper-Local-Retail-Mobility-Analytics\.venv\Scripts\python.exe


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("GeoPulse Environment Ready!")

GeoPulse Environment Ready!


In [3]:
import os

print(os.getcwd())
print(os.listdir())

C:\Users\HP\GeoPulse--Hyper-Local-Retail-Mobility-Analytics
['.git', '.ipynb_checkpoints', '.venv', 'data', 'dbt', 'notebooks', 'README.md', 'reports', 'sql', 'src', 'Untitled.ipynb']


In [4]:
import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 3.1.1
NumPy: 2.5.4


In [5]:
import pandas as pd
import numpy as np

np.random.seed(42)

data = {
    "DeviceID": [f"D{i:03d}" for i in range(1, 101)],
    "Lat": np.random.uniform(28.60, 28.70, 100),
    "Long": np.random.uniform(77.10, 77.25, 100),
    "Timestamp": pd.date_range(
        start="2026-11-01",
        periods=100,
        freq="h"
    )
}

gps_data = pd.DataFrame(data)

gps_data.head()

,DeviceID,Lat,Long,Timestamp
0,D001,28.637454,77.104714,2026-11-01 00:00:00
1,D002,28.695071,77.195462,2026-11-01 01:00:00
2,D003,28.673199,77.147153,2026-11-01 02:00:00
3,D004,28.659866,77.176286,2026-11-01 03:00:00
4,D005,28.615602,77.236135,2026-11-01 04:00:00


In [6]:
gps_data.shape

(100, 4)

In [7]:
gps_data.columns

Index(['DeviceID', 'Lat', 'Long', 'Timestamp'], dtype='str')

In [8]:
gps_data.dtypes

DeviceID                str
Lat                 float64
Long                float64
Timestamp    datetime64[us]
dtype: object

In [9]:
gps_data.isnull().sum()

DeviceID     0
Lat          0
Long         0
Timestamp    0
dtype: int64

In [10]:
gps_data.duplicated().sum()

np.int64(0)

In [11]:
gps_data.describe()

,Lat,Long,Timestamp
count,100.000000,100.000000,100
mean,28.647018,77.174675,2026-11-03 01:30:00
min,28.600552,77.101043,2026-11-01 00:00:00
25%,28.619320,77.136301,2026-11-02 00:45:00
50%,28.646414,77.175844,2026-11-03 01:30:00
75%,28.673020,77.214928,2026-11-04 02:15:00
max,28.698689,77.247848,2026-11-05 03:00:00
std,0.029749,0.043967,NaN


In [12]:
gps_data.to_csv("geopulse_gps_data.csv", index=False)

print("GPS Dataset Saved Successfully!")

GPS Dataset Saved Successfully!


In [13]:
gps_check = pd.read_csv("geopulse_gps_data.csv")

print("Total Records:", len(gps_check))
print("Missing Values:")
print(gps_check.isnull().sum())

print("Duplicate Records:", gps_check.duplicated().sum())

Total Records: 100
Missing Values:
DeviceID     0
Lat          0
Long         0
Timestamp    0
dtype: int64
Duplicate Records: 0


In [14]:
print("Latitude Range:")
print(gps_check["Lat"].min(), gps_check["Lat"].max())

print("Longitude Range:")
print(gps_check["Long"].min(), gps_check["Long"].max())

Latitude Range:
28.600552211712362 28.698688693660053
Longitude Range:
77.10104281957967 77.24784756811658


In [15]:
stores = pd.DataFrame({
    "StoreID": ["S001", "S002", "S003", "S004", "S005"],
    "StoreName": [
        "Store A", "Store B", "Store C",
        "Store D", "Store E"
    ],
    "Lat": [28.6500, 28.6700, 28.6300, 28.6900, 28.6100],
    "Long": [77.1500, 77.1800, 77.2000, 77.1200, 77.2300]
})

stores

,StoreID,StoreName,Lat,Long
0,S001,Store A,28.65,77.15
1,S002,Store B,28.67,77.18
2,S003,Store C,28.63,77.20
3,S004,Store D,28.69,77.12
4,S005,Store E,28.61,77.23


In [16]:
stores.to_csv("geopulse_store_locations.csv", index=False)

print("Store Dataset Saved Successfully!")

Store Dataset Saved Successfully!


In [17]:
%pip install geopy


   ---------------------------------------- 0/2 [geographiclib]
   ---------------------------------------- 0/2 [geographiclib]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   ---------------------------------------- 2/2 [geopy]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from geopy.distance import geodesic

for i, store in stores.iterrows():

    count = 0

    for j, gps in gps_data.iterrows():

        distance = geodesic(
            (store["Lat"], store["Long"]),
            (gps["Lat"], gps["Long"])
        ).meters

        if distance <= 500:
            count += 1

    print(store["StoreName"], ":", count, "devices within 500 meters")

Store A : 1 devices within 500 meters
Store B : 0 devices within 500 meters
Store C : 1 devices within 500 meters
Store D : 1 devices within 500 meters
Store E : 1 devices within 500 meters


In [19]:
catchment_results = []

for i, store in stores.iterrows():
    count = 0

    for j, gps in gps_data.iterrows():
        distance = geodesic(
            (store["Lat"], store["Long"]),
            (gps["Lat"], gps["Long"])
        ).meters

        if distance <= 500:
            count += 1

    catchment_results.append({
        "StoreID": store["StoreID"],
        "StoreName": store["StoreName"],
        "Devices_Within_500m": count
    })

catchment_df = pd.DataFrame(catchment_results)

catchment_df

,StoreID,StoreName,Devices_Within_500m
0,S001,Store A,1
1,S002,Store B,0
2,S003,Store C,1
3,S004,Store D,1
4,S005,Store E,1


In [20]:
catchment_df.to_csv(
    "geopulse_catchment_analysis.csv",
    index=False
)

print("Catchment Analysis Saved Successfully!")

Catchment Analysis Saved Successfully!


In [1]:
import sys
import os

print("Python Executable:", sys.executable)
print("Python Version:", sys.version)
print("Working Directory:", os.getcwd())

Python Executable: C:\ProgramData\anaconda3\python.exe
Python Version: 3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]
Working Directory: C:\Users\visha\Desktop\GeoPulse--Hyper-Local-Retail-Mobility-Analytics


In [2]:
libraries = [
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "geopy",
    "geopandas",
    "shapely"
]

for library in libraries:
    try:
        module = __import__(library)
        print(f"{library}: Installed ({getattr(module, '__version__', 'Version unknown')})")
    except ImportError:
        print(f"{library}: Not Installed")

pandas: Installed (3.0.3)
numpy: Installed (2.4.6)
matplotlib: Installed (3.11.0)
seaborn: Installed (0.13.2)
geopy: Not Installed
geopandas: Not Installed
shapely: Not Installed


In [ ]:
where python
where jupyter
jupyter kernelspec list